In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Optimizări de transpilare cu SABRE

import TutorialFeedback from '@site/src/components/TutorialFeedback';





*Estimare utilizare: sub un minut pe un procesor Heron r2 (NOTĂ: Aceasta este doar o estimare. Timpul tău de execuție poate varia.)*
## Context
Transpilarea este un pas esențial în Qiskit care convertește circuitele cuantice în forme compatibile cu hardware-ul cuantic specific. Implică două etape cheie: **layout-ul qubiților** (maparea qubiților logici pe qubiții fizici de pe dispozitiv) și **rutarea porților** (asigurarea că porțile multi-qubit respectă conectivitatea dispozitivului prin inserarea de porți SWAP după necesitate).

SABRE (*SWAP-Based Bidirectional heuristic search algorithm* — algoritm euristic bidirecțional bazat pe SWAP) este un instrument puternic de optimizare atât pentru layout, cât și pentru rutare. Este deosebit de eficient pentru **circuite la scară largă** (100+ qubiți) și dispozitive cu hărți de cuplare complexe, precum **IBM&reg; Heron**, unde creșterea exponențială a posibilelor mapări de qubiți necesită soluții eficiente.

### De ce să folosești SABRE?
SABRE minimizează numărul de porți SWAP și reduce adâncimea circuitului, îmbunătățind performanța circuitului pe hardware-ul real. Abordarea sa bazată pe euristici o face ideală pentru hardware avansat și circuite mari și complexe. Îmbunătățirile recente introduse în algoritmul [LightSABRE](https://arxiv.org/abs/2409.08368) optimizează și mai mult performanța SABRE, oferind timpi de execuție mai rapizi și mai puține porți SWAP. Aceste îmbunătățiri îl fac și mai eficient pentru circuitele la scară largă.

### Ce vei învăța
Acest tutorial este împărțit în două părți:
1. Înveți să folosești SABRE cu **Qiskit patterns** pentru optimizarea avansată a circuitelor mari.
2. Valorifici **qiskit_serverless** pentru a maximiza potențialul SABRE în transpilarea scalabilă și eficientă.

Vei:
- Optimiza SABRE pentru circuite cu 100+ qubiți, depășind setările implicite de transpilare precum `optimization_level=3`.
- Explora **îmbunătățirile LightSABRE** care îmbunătățesc timpul de execuție și reduc numărul de porți.
- Personaliza parametrii cheie SABRE (`swap_trials`, `layout_trials`, `max_iterations`, `heuristic`) pentru a echilibra **calitatea circuitului** și **timpul de transpilare**.
## Cerințe
Înainte de a începe acest tutorial, asigură-te că ai instalat următoarele:
- Qiskit SDK v1.0 sau mai recent, cu suport pentru [vizualizare](https://docs.quantum.ibm.com/api/qiskit/visualization)
- Qiskit Runtime v0.28 sau mai recent (`pip install qiskit-ibm-runtime`)
- Serverless (`pip install qiskit-ibm-catalog qiskit_serverless`)
## Configurare

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorOptions
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_aer.primitives import EstimatorV2 as AerEstimator
from qiskit.transpiler.passes import (
    SabreLayout,
    SabreSwap,
    BarrierBeforeFinalMeasurements,
    StarPreRouting,
)
from qiskit.transpiler.passes.layout.vf2_layout import VF2LayoutStopReason
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.passmanager.flow_controllers import ConditionalController
import matplotlib.pyplot as plt
import numpy as np
import time

seed = 42

service = QiskitRuntimeService(
    channel="ibm_cloud",
    token="<YOUR_API_TOKEN>",  # Replace with your actual API token
    instance="<YOUR_INSTANCE_NAME>",  # Replace with your instance name if needed
)
backend = service.least_busy(operational=True, simulator=False)


print(f"Using backend: {backend.name}")

Using backend: ibm_kingston


## Partea I. Utilizarea SABRE cu Qiskit patterns

SABRE poate fi utilizat în Qiskit pentru a optimiza circuitele cuantice gestionând atât etapele de layout al qubiților, cât și de rutare a porților. În această secțiune, te vom ghida prin **exemplul minimal** de utilizare a SABRE cu Qiskit patterns, cu focalizare principală pe pasul 2 al optimizării.

Pentru a rula SABRE, ai nevoie de:
- O reprezentare **DAG** (Graf Aciclic Direcționat) a circuitului tău cuantic.
- **Harta de cuplare** de la backend, care specifică modul în care qubiții sunt conectați fizic.
- **Pasul SABRE**, care aplică algoritmul pentru a optimiza layout-ul și rutarea.

Pentru această parte, ne vom concentra pe pasul **SabreLayout**. Acesta efectuează atât încercări de layout, cât și de rutare, lucrând pentru a găsi cel mai eficient layout inițial, minimizând în același timp numărul de porți SWAP necesare. Important de menționat, `SabreLayout`, prin el însuși, optimizează intern atât layout-ul, cât și rutarea, stocând soluția care adaugă cel mai mic număr de porți SWAP. Rețineți că atunci când se utilizează doar **SabreLayout**, nu putem schimba euristica SABRE, dar putem personaliza numărul de `layout_trials`.

### Pasul 1: Mapează intrările clasice la o problemă cuantică

Un circuit **GHZ (Greenberger-Horne-Zeilinger)** este un circuit cuantic care pregătește o stare întreținută în care toți qubiții sunt fie în starea `|0...0⟩`, fie în starea `|1...1⟩`. Starea GHZ pentru $n$ qubiți este reprezentată matematic astfel:
$$ |\text{GHZ}\rangle = \frac{1}{\sqrt{2}} \left( |0\rangle^{\otimes n} + |1\rangle^{\otimes n} \right) $$

Este construit prin aplicarea:
1. Unui Gate Hadamard primului qubit pentru a crea superpozitie.
2. Unei serii de porți CNOT pentru a întrețese qubiții rămași cu primul.

Pentru acest exemplu, construim în mod deliberat un **Circuit GHZ cu topologie stea** în loc de unul cu topologie liniară. În topologia stea, primul qubit acționează ca „hub", iar toți ceilalți qubiți sunt întrețesuți direct cu el folosind porți CNOT. Această alegere este deliberată deoarece, în timp ce **starea GHZ cu topologie liniară** poate fi teoretic implementată în adâncime $ O(N) $ pe o hartă de cuplare liniară fără nicio poartă SWAP, SABRE ar găsi trivial o soluție optimă prin maparea unui Circuit GHZ de 100 de qubiți pe un subgraf al hărții de cuplare heavy-hex a Backend-ului.

**Circuitul GHZ cu topologie stea** pune o problemă semnificativ mai dificilă. Deși poate fi executat teoretic în adâncime $ O(N) $ fără porți SWAP, găsirea acestei soluții necesită identificarea unui layout inițial optim, ceea ce este mult mai dificil din cauza conectivității non-liniare a circuitului. Această topologie servește ca un caz de testare mai bun pentru evaluarea SABRE, deoarece demonstrează modul în care parametrii de configurare influențează performanța layout-ului și rutării în condiții mai complexe.

![ghz_star_topology.png](../docs/images/tutorials/transpilation-optimizations-with-sabre/ghz_star_topology.avif)

De remarcat:
- Instrumentul **HighLevelSynthesis** poate produce soluția optimă de adâncime $ O(N) $ pentru Circuitul GHZ cu topologie stea fără a introduce porți SWAP, așa cum se arată în imaginea de mai sus.
- Alternativ, pasul **StarPrerouting** poate reduce și mai mult adâncimea ghidând deciziile de rutare ale SABRE, deși poate introduce în continuare unele porți SWAP. Cu toate acestea, StarPrerouting crește timpul de execuție și necesită integrare în procesul inițial de transpilare.

În scopul acestui tutorial, excludem atât HighLevelSynthesis, cât și StarPrerouting pentru a izola și evidenția impactul direct al configurației SABRE asupra timpului de execuție și adâncimii circuitului. Prin măsurarea valorii de așteptare $ \langle Z_0 Z_i \rangle $ pentru fiecare pereche de qubiți, analizăm:
- Cât de bine reduce SABRE porțile SWAP și adâncimea circuitului.
- Efectul acestor optimizări asupra fidelității circuitului executat, unde abaterile de la $ \langle Z_0 Z_i \rangle = 1 $ indică pierderea întreținerii.!

In [2]:
num_qubits_sim = 15

# Create star-topology GHZ circuit
qc_sim = QuantumCircuit(num_qubits_sim)
qc_sim.h(0)
for i in range(1, num_qubits_sim):
    qc_sim.cx(0, i)
qc_sim.measure_all()

# ZZ operators: Z on qubit 0 and qubit i, identity elsewhere
operator_strings_sim = [
    "Z" + "I" * i + "Z" + "I" * (num_qubits_sim - 2 - i)
    for i in range(num_qubits_sim - 1)
]
operators_sim = [SparsePauliOp(op) for op in operator_strings_sim]

În continuare, vom mapa operatorii de interes pentru a evalua comportamentul sistemului. Vom utiliza operatorii `ZZ` dintre qubiți pentru a examina modul în care întrețeserea se degradează pe măsură ce qubiții se îndepărtează. Această analiză este esențială deoarece inexactitățile în valorile de așteptare $\langle Z_0 Z_i \rangle$ pentru qubiții îndepărtați pot releva impactul zgomotului și al erorilor în execuția circuitului. Studiind aceste abateri, obținem o perspectivă asupra modului în care circuitul păstrează întrețeserea sub diferite configurații SABRE și cât de eficient minimizează SABRE impactul constrângerilor hardware.

In [3]:
# Build the default pass manager (no modifications yet)
pm_1 = generate_preset_pass_manager(
    optimization_level=3, backend=backend, seed_transpiler=seed
)

# Visualize the layout stage to see where SabreLayout sits
pm_1.layout.draw()

<Image src="../docs/images/tutorials/transpilation-optimizations-with-sabre/extracted-outputs/b40fe1e0-41cd-4e8b-acb9-801872d35f1f-0.avif" alt="Output of the previous code cell" />

In the diagram above, the `SabreLayout` pass we want to customize lives inside the `ConditionalController` at position **[2]** of the layout stage. That controller does two things:

- It gates `SabreLayout` so it only runs when `VF2Layout` at [1] failed to find a perfect mapping (otherwise the perfect VF2 layout is kept).
- It precedes `SabreLayout` with a `BarrierBeforeFinalMeasurements` pass that protects measurements from being reordered during SabreLayout's internal routing.

If we just `replace(index=2, passes=sl_2)`, both behaviors are dropped. To keep them, we re-wrap our custom `SabreLayout` in the same `ConditionalController` (with the same condition and the protective barrier) before swapping it in.

**Step 2b: Build custom `SabreLayout` passes and replace the default.**

In [4]:
cmap = backend.coupling_map

# Custom SabreLayout passes with more aggressive search
sl_2 = SabreLayout(
    coupling_map=cmap,
    seed=seed,
    max_iterations=4,
    layout_trials=200,
    swap_trials=200,
)
sl_3 = SabreLayout(
    coupling_map=cmap,
    seed=seed,
    max_iterations=8,
    layout_trials=200,
    swap_trials=200,
)


# Same condition the preset uses: only run SabreLayout when VF2Layout did not
# find a perfect mapping. This preserves any perfect layout VF2 produced at [1].
def _vf2_match_not_found(property_set):
    if property_set["layout"] is None:
        return True
    return (
        property_set["VF2Layout_stop_reason"] is not None
        and property_set["VF2Layout_stop_reason"]
        is not VF2LayoutStopReason.SOLUTION_FOUND
    )


def wrap_sabre(sabre_pass):
    """Re-wrap a SabreLayout in the original ConditionalController + barrier."""
    return ConditionalController(
        [
            BarrierBeforeFinalMeasurements(
                "qiskit.transpiler.internal.routing.protection.barrier"
            ),
            sabre_pass,
        ],
        condition=_vf2_match_not_found,
    )


# Build two fresh pass managers and swap in the wrapped custom SabreLayout at index 2
pm_2 = generate_preset_pass_manager(
    optimization_level=3, backend=backend, seed_transpiler=seed
)
pm_3 = generate_preset_pass_manager(
    optimization_level=3, backend=backend, seed_transpiler=seed
)
pm_2.layout.replace(index=2, passes=wrap_sabre(sl_2))
pm_3.layout.replace(index=2, passes=wrap_sabre(sl_3))

# Build pm_star: default preset with StarPreRouting added to the init stage
pm_star = generate_preset_pass_manager(
    optimization_level=3, backend=backend, seed_transpiler=seed
)
pm_star.init += StarPreRouting()

# Visualize pm_3 after replacement (pm_2 has the same structure, only max_iterations differs)
pm_3.layout.draw()

<Image src="../docs/images/tutorials/transpilation-optimizations-with-sabre/extracted-outputs/79075a21-8f36-4fd9-9d0d-bd0e97395b60-0.avif" alt="Output of the previous code cell" />

### Pasul 2: Optimizează problema pentru execuția pe hardware cuantic
În acest pas, ne concentrăm pe optimizarea layout-ului circuitului pentru execuția pe un dispozitiv specific de hardware cuantic cu 127 de qubiți. Acesta este focusul principal al tutorialului, deoarece efectuăm **optimizările SABRE și transpilarea** pentru a obține cea mai bună performanță a circuitului. Folosind pasul `SabreLayout`, determinăm o mapare inițială a qubiților care minimizează nevoia de porți SWAP în timpul rutării. Prin furnizarea `coupling_map` a Backend-ului țintă, `SabreLayout` adaptează layout-ul la constrângerile de conectivitate ale dispozitivului.

Vom folosi `generate_preset_pass_manager` cu `optimization_level=3` pentru procesul de transpilare și vom personaliza pasul `SabreLayout` cu diferite configurații. Obiectivul este de a găsi o configurație care produce un circuit transpilat cu **cea mai mică dimensiune și/sau adâncime**, demonstrând impactul optimizărilor SABRE.

#### De ce sunt importante dimensiunea și adâncimea circuitului?
- **Dimensiune mai mică (număr de porți):** Reduce numărul de operații, minimizând oportunitățile de acumulare a erorilor.
- **Adâncime mai mică:** Scurtează timpul total de execuție, critic pentru evitarea decoherenței și menținerea fidelității stării cuantice.

Prin optimizarea acestor metrici, îmbunătățim fiabilitatea circuitului și acuratețea execuției pe hardware-ul cuantic cu zgomot.
Selectează Backend-ul.

In [5]:
results_sim = {}
for name, pm in [
    ("pm_1 (4,20,20)", pm_1),
    ("pm_2 (4,200,200)", pm_2),
    ("pm_3 (8,200,200)", pm_3),
    ("pm_star (default + StarPreRouting)", pm_star),
]:
    t0 = time.time()
    tqc = pm.run(qc_sim)
    elapsed = time.time() - t0
    depth = tqc.depth(lambda x: x.operation.num_qubits == 2)
    size = tqc.size()
    ops_mapped = [op.apply_layout(tqc.layout) for op in operators_sim]
    results_sim[name] = {
        "tqc": tqc,
        "ops": ops_mapped,
        "depth": depth,
        "size": size,
        "time": elapsed,
    }
    print(f"{name}: 2Q Depth {depth}, Size {size}, Time {elapsed:.2f}s")

# Print improvement relative to default (pm_1)
baseline = results_sim["pm_1 (4,20,20)"]
print("\nImprovement vs. default (pm_1):")
for name in [
    "pm_2 (4,200,200)",
    "pm_3 (8,200,200)",
    "pm_star (default + StarPreRouting)",
]:
    r = results_sim[name]
    depth_pct = (baseline["depth"] - r["depth"]) / baseline["depth"] * 100
    size_pct = (baseline["size"] - r["size"]) / baseline["size"] * 100
    print(f"  {name}: 2Q depth {depth_pct:+.1f}%, size {size_pct:+.1f}%")

pm_1 (4,20,20): 2Q Depth 38, Size 183, Time 0.01s
pm_2 (4,200,200): 2Q Depth 36, Size 183, Time 0.15s
pm_3 (8,200,200): 2Q Depth 30, Size 158, Time 0.16s
pm_star (default + StarPreRouting): 2Q Depth 26, Size 160, Time 0.01s

Improvement vs. default (pm_1):
  pm_2 (4,200,200): 2Q depth +5.3%, size +0.0%
  pm_3 (8,200,200): 2Q depth +21.1%, size +13.7%
  pm_star (default + StarPreRouting): 2Q depth +31.6%, size +12.6%


All three modified pass managers produced circuits with lower 2Q depth than the default. The aggressive SABRE configurations (`pm_2` and `pm_3`) trade longer transpilation time for a wider search, while `pm_star` leverages the star structure of the circuit and produces an even shallower result without paying any extra transpilation cost. The exact gains will vary from run to run, but the general trend is consistent: more SABRE trials and iterations let the heuristic search a wider space, and structure-aware passes like `StarPreRouting` can sidestep that search entirely when the circuit shape matches.

Even at this small scale (15 qubits), the room for improvement is enough that all three approaches beat the default. With larger circuits (100+ qubits), the search space grows dramatically and the benefits of both increased trials and structure-aware passes become much more pronounced, as the large-scale section will show.

In [6]:
pm_names = list(results_sim.keys())
depths = [results_sim[n]["depth"] for n in pm_names]
sizes = [results_sim[n]["size"] for n in pm_names]
times = [results_sim[n]["time"] for n in pm_names]
colors = ["#404080", "#2a9d8f", "#a8d05e", "#e29bdd"]
x = np.arange(len(pm_names))

fig, axs = plt.subplots(1, 3, figsize=(14, 5))

# 2Q Depth
bars = axs[0].bar(x, depths, color=colors)
axs[0].set_ylabel("2Q Depth", fontsize=11)
axs[0].set_title("Two-Qubit Gate Depth", fontsize=13)
axs[0].set_ylim(0, max(depths) * 1.2)
for bar, val in zip(bars, depths):
    axs[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(depths) * 0.02,
        str(val),
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold",
    )
for i in range(1, len(depths)):
    pct = (depths[0] - depths[i]) / depths[0] * 100
    if pct != 0:
        axs[0].text(
            bars[i].get_x() + bars[i].get_width() / 2,
            bars[i].get_height() / 2,
            f"{pct:+.0f}%",
            ha="center",
            va="center",
            fontsize=10,
            color="white",
            fontweight="bold",
        )

# Size
bars = axs[1].bar(x, sizes, color=colors)
axs[1].set_ylabel("Gate Count", fontsize=11)
axs[1].set_title("Circuit Size", fontsize=13)
axs[1].set_ylim(0, max(sizes) * 1.2)
for bar, val in zip(bars, sizes):
    axs[1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(sizes) * 0.02,
        str(val),
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold",
    )
for i in range(1, len(sizes)):
    pct = (sizes[0] - sizes[i]) / sizes[0] * 100
    if abs(pct) > 0.1:
        axs[1].text(
            bars[i].get_x() + bars[i].get_width() / 2,
            bars[i].get_height() / 2,
            f"{pct:+.0f}%",
            ha="center",
            va="center",
            fontsize=10,
            color="white",
            fontweight="bold",
        )

# Time
bars = axs[2].bar(x, times, color=colors)
axs[2].set_ylabel("Time (s)", fontsize=11)
axs[2].set_title("Transpilation Time", fontsize=13)
axs[2].set_ylim(0, max(times) * 1.3)
for bar, val in zip(bars, times):
    axs[2].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(times) * 0.03,
        f"{val:.2f}s",
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold",
    )

for ax in axs:
    ax.set_xticks(x)
    ax.set_xticklabels(pm_names, fontsize=8, rotation=15)
    ax.grid(axis="y", linestyle="--", alpha=0.5)

plt.suptitle(
    "Transpilation quality vs. configuration",
    fontsize=14,
    fontweight="bold",
    y=1.02,
)
plt.tight_layout()
plt.show()

<Image src="../docs/images/tutorials/transpilation-optimizations-with-sabre/extracted-outputs/bf75c45a-2c3e-4ef6-8336-0b3f69e6e8fb-0.avif" alt="Output of the previous code cell" />

Pentru a evalua impactul diferitelor configurații asupra optimizării circuitului, vom crea trei manageri de pași, fiecare cu setări unice pentru pasul `SabreLayout`. Aceste configurații ajută la analizarea compromisului dintre calitatea circuitului și timpul de transpilare.

#### Parametri cheie
- **`max_iterations`**: Numărul de iterații de rutare înainte-înapoi pentru rafinarea layout-ului și reducerea costurilor de rutare.
- **`layout_trials`**: Numărul de layout-uri inițiale aleatoare testate, selectând cel care minimizează porțile SWAP.
- **`swap_trials`**: Numărul de încercări de rutare pentru fiecare layout, rafinând plasarea porților pentru o rutare mai bună.

Crește `layout_trials` și `swap_trials` pentru o optimizare mai riguroasă, cu costul unui timp de transpilare crescut.

#### Configurații în acest tutorial
1. **`pm_1`**: Setări implicite cu `optimization_level=3`.
   - `max_iterations=4`
   - `layout_trials=20`
   - `swap_trials=20`

2. **`pm_2`**: Crește numărul de încercări pentru o explorare mai bună.
   - `max_iterations=4`
   - `layout_trials=200`
   - `swap_trials=200`

3. **`pm_3`**: Extinde `pm_2` prin creșterea numărului de iterații pentru rafinare suplimentară.
   - `max_iterations=8`
   - `layout_trials=200`
   - `swap_trials=200`

Prin compararea rezultatelor acestor configurații, urmărim să determinăm care obține cel mai bun echilibru între calitatea circuitului (de exemplu, dimensiunea și adâncimea) și costul computațional.

In [15]:
# Create a noisy estimator from the real backend's noise model
noisy_estimator = AerEstimator.from_backend(backend)

num_runs = 10
# sim_all_runs[name] = list of arrays, one per run
sim_all_runs = {name: [] for name in results_sim}

for run in range(num_runs):
    for name, r in results_sim.items():
        job = noisy_estimator.run([(r["tqc"], r["ops"])])
        evs = list(job.result()[0].data.evs)
        sim_all_runs[name].append(evs)
    print(f"Run {run + 1}/{num_runs} done")

# Compute mean and std across runs for each config
sim_stats = {}
for name in results_sim:
    all_evs = np.array(sim_all_runs[name])  # shape (num_runs, num_operators)
    sim_stats[name] = {
        "mean": np.mean(all_evs, axis=0),
        "std": np.std(all_evs, axis=0),
        "overall_mean": np.mean(all_evs),
        "overall_std": np.std(
            np.mean(all_evs, axis=1)
        ),  # std of per-run averages
    }
    print(
        f"{name}: mean fidelity = {sim_stats[name]['overall_mean']:.4f} +/- {sim_stats[name]['overall_std']:.4f}"
    )

Run 1/10 done
Run 2/10 done
Run 3/10 done
Run 4/10 done
Run 5/10 done
Run 6/10 done
Run 7/10 done
Run 8/10 done
Run 9/10 done
Run 10/10 done
pm_1 (4,20,20): mean fidelity = 0.9510 +/- 0.0094
pm_2 (4,200,200): mean fidelity = 0.9513 +/- 0.0043
pm_3 (8,200,200): mean fidelity = 0.9540 +/- 0.0065
pm_star (default + StarPreRouting): mean fidelity = 0.9547 +/- 0.0072


Acum putem configura pasul `SabreLayout` în managerii de pași personalizați. Pentru aceasta, știm că pentru `generate_preset_pass_manager` implicit la `optimization_level=3`, pasul `SabreLayout` se află la indexul 2, deoarece `SabreLayout` apare după pasele `SetLayout` și `VF2Layout`. Putem accesa acest pas și modifica parametrii săi.

In [16]:
data_sim = list(range(1, len(operators_sim) + 1))
markers = ["o", "s", "^", "*"]
colors_line = ["#404080", "#2a9d8f", "#a8d05e", "#e29bdd"]

fig, (ax1, ax2) = plt.subplots(
    1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [2.5, 1]}
)

# Left: correlations vs distance with error bars (mean +/- 1 std)
for (name, stats), marker, color in zip(
    sim_stats.items(), markers, colors_line
):
    ax1.errorbar(
        data_sim,
        stats["mean"],
        yerr=stats["std"],
        marker=marker,
        label=name,
        color=color,
        linewidth=2,
        capsize=3,
        capthick=1,
        elinewidth=1,
    )

ax1.set_xlabel("Distance between qubits $i$", fontsize=11)
ax1.set_ylabel(r"$\langle Z_0 Z_i \rangle$", fontsize=11)
ax1.set_title(
    "Entanglement correlations vs. qubit distance (avg. of 10 runs)",
    fontsize=12,
)
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

# Right: mean correlation bar chart with error bars
names = list(sim_stats.keys())
means = [sim_stats[n]["overall_mean"] for n in names]
stds = [sim_stats[n]["overall_std"] for n in names]
x_bar = np.arange(len(names))
bars = ax2.bar(
    x_bar, means, yerr=stds, color=colors_line, capsize=5, ecolor="gray"
)
ax2.set_ylabel(r"Mean $\langle Z_0 Z_i \rangle$", fontsize=11)
ax2.set_title("Average fidelity", fontsize=13, pad=12)
y_range = max(means) - min(means) if max(means) != min(means) else 0.01
# Top of ylim accounts for the bar height + std error bar + headroom for the value label
y_top = max(m + s for m, s in zip(means, stds)) + y_range * 1.5
ax2.set_ylim(min(means) - y_range * 0.8, y_top)
for bar, val, std in zip(bars, means, stds):
    ax2.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + std + y_range * 0.15,
        f"{val:.4f}",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold",
    )
# Annotate % change vs pm_1
baseline_mean = means[0]
for i in range(1, len(means)):
    pct = (means[i] - baseline_mean) / baseline_mean * 100
    if abs(pct) > 0.01:
        mid_y = (means[i] + ax2.get_ylim()[0]) / 2
        ax2.text(
            bars[i].get_x() + bars[i].get_width() / 2,
            mid_y,
            f"{pct:+.1f}%",
            ha="center",
            va="center",
            fontsize=10,
            color="white",
            fontweight="bold",
        )
ax2.set_xticks(x_bar)
ax2.set_xticklabels(names, fontsize=8, rotation=15)
ax2.grid(axis="y", linestyle="--", alpha=0.5)

fig.tight_layout()
plt.show()

<Image src="../docs/images/tutorials/transpilation-optimizations-with-sabre/extracted-outputs/a6dac5ed-a963-458a-ada1-89c915f036e0-0.avif" alt="Output of the previous code cell" />

Cu fiecare manager de pași configurat, vom executa acum procesul de transpilare pentru fiecare. Pentru a compara rezultatele, vom urmări metrici cheie, inclusiv timpul de transpilare, adâncimea circuitului (măsurată ca adâncimea porților cu doi qubiți) și numărul total de porți din circuitele transpilate.

In [9]:
# -------------------------Step 1-------------------------

num_qubits = 100

# Create star-topology GHZ circuit
qc = QuantumCircuit(num_qubits)
qc.h(0)
for i in range(1, num_qubits):
    qc.cx(0, i)
qc.measure_all()

# ZZ operators
operator_strings = [
    "Z" + "I" * i + "Z" + "I" * (num_qubits - 2 - i)
    for i in range(num_qubits - 1)
]
operators = [SparsePauliOp(op) for op in operator_strings]

In [10]:
# -------------------------Step 2-------------------------

num_seeds = 10
seed_list = [seed + i for i in range(num_seeds)]
swap_trials = 200


# The default routing[1] is a ConditionalController([barrier, routing_pass],
# condition=_swap_condition); we re-wrap so the new routing pass keeps the
# protective barrier and is skipped when routing isn't needed (matches the preset).
def _swap_condition(property_set):
    return not property_set["routing_not_needed"]


def wrap_routing(routing_pass):
    return ConditionalController(
        [
            BarrierBeforeFinalMeasurements(
                "qiskit.transpiler.internal.routing.protection.barrier"
            ),
            routing_pass,
        ],
        condition=_swap_condition,
    )


heuristic_results = {}

# Three SABRE heuristics, swept over seeds
for heuristic in ["basic", "decay", "lookahead"]:
    trials = []
    for s in seed_list:
        sr = SabreSwap(
            coupling_map=cmap, heuristic=heuristic, trials=swap_trials, seed=s
        )
        sl = SabreLayout(coupling_map=cmap, routing_pass=sr, seed=s)
        pm = generate_preset_pass_manager(
            optimization_level=3, backend=backend, seed_transpiler=s
        )
        # Re-wrap each custom pass in its original ConditionalController + barrier
        # (wrap_sabre is defined in the small-scale Step 2 cell above).
        pm.layout.replace(index=2, passes=wrap_sabre(sl))
        pm.routing.replace(index=1, passes=wrap_routing(sr))

        t0 = time.time()
        tqc = pm.run(qc)
        elapsed = time.time() - t0
        depth = tqc.depth(lambda x: x.operation.num_qubits == 2)
        size = tqc.size()
        trials.append(
            {
                "tqc": tqc,
                "depth": depth,
                "size": size,
                "time": elapsed,
                "seed": s,
            }
        )

    heuristic_results[heuristic] = trials

# Default preset + StarPreRouting in init, also swept over seeds for a fair comparison
star_trials = []
for s in seed_list:
    pm_star_hw = generate_preset_pass_manager(
        optimization_level=3, backend=backend, seed_transpiler=s
    )
    pm_star_hw.init += StarPreRouting()

    t0 = time.time()
    tqc = pm_star_hw.run(qc)
    elapsed = time.time() - t0
    depth = tqc.depth(lambda x: x.operation.num_qubits == 2)
    size = tqc.size()
    star_trials.append(
        {
            "tqc": tqc,
            "depth": depth,
            "size": size,
            "time": elapsed,
            "seed": s,
        }
    )
heuristic_results["StarPreRouting"] = star_trials

# Print summary for each entry
for label in ["basic", "decay", "lookahead", "StarPreRouting"]:
    trials = heuristic_results[label]
    depths = [t["depth"] for t in trials]
    sizes = [t["size"] for t in trials]
    best = min(trials, key=lambda t: t["depth"])
    print(f"{label}:")
    print(
        f"  2Q depth: min: {min(depths)}, mean: {np.mean(depths):.1f}, std: {np.std(depths):.1f}"
    )
    print(
        f"  size    : min: {min(sizes)}, mean: {np.mean(sizes):.1f}, std: {np.std(sizes):.1f}"
    )
    print(
        f"  best seed: {best['seed']} (2Q depth={best['depth']}, size={best['size']})"
    )

basic:
  2Q depth: min: 524, mean: 570.5, std: 39.9
  size    : min: 3819, mean: 4227.1, std: 360.6
  best seed: 51 (2Q depth=524, size=3852)
decay:
  2Q depth: min: 387, mean: 436.4, std: 41.7
  size    : min: 2687, mean: 3183.1, std: 459.3
  best seed: 45 (2Q depth=387, size=2786)
lookahead:
  2Q depth: min: 364, mean: 424.6, std: 36.5
  size    : min: 2335, mean: 3014.6, std: 388.1
  best seed: 51 (2Q depth=364, size=2485)
StarPreRouting:
  2Q depth: min: 196, mean: 196.0, std: 0.0
  size    : min: 1151, mean: 1151.0, std: 0.0
  best seed: 42 (2Q depth=196, size=1151)


In [11]:
hw_colors = {
    "basic": "#ff7f0e",
    "decay": "#d62728",
    "lookahead": "#1f77b4",
    "StarPreRouting": "#2a9d8f",
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

for label in ["basic", "decay", "lookahead", "StarPreRouting"]:
    trials = heuristic_results[label]
    depths = [t["depth"] for t in trials]
    sizes = [t["size"] for t in trials]
    seeds = [t["seed"] for t in trials]
    color = hw_colors[label]

    ax1.scatter(
        seeds,
        depths,
        label=label,
        color=color,
        alpha=0.8,
        edgecolor="k",
        s=60,
    )
    ax1.axhline(np.mean(depths), color=color, linestyle="--", alpha=0.5)

    ax2.scatter(
        seeds,
        sizes,
        label=label,
        color=color,
        alpha=0.8,
        edgecolor="k",
        s=60,
    )
    ax2.axhline(np.mean(sizes), color=color, linestyle="--", alpha=0.5)

ax1.set_xlabel("Seed", fontsize=11)
ax1.set_ylabel("2Q Depth", fontsize=11)
ax1.set_title("Two-Qubit Gate Depth per Seed", fontsize=13)
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)

ax2.set_xlabel("Seed", fontsize=11)
ax2.set_ylabel("Gate Count", fontsize=11)
ax2.set_title("Circuit Size per Seed", fontsize=13)
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

plt.suptitle(
    "Transpilation variability across seeds: SABRE heuristics vs. StarPreRouting",
    fontsize=14,
    fontweight="bold",
    y=1.02,
)
plt.tight_layout()
plt.show()

# Summary comparison
for label in ["basic", "decay", "lookahead", "StarPreRouting"]:
    best = min(heuristic_results[label], key=lambda t: t["depth"])
    print(
        f"{label}: best 2Q depth={best['depth']}, size={best['size']} (seed={best['seed']})"
    )

<Image src="../docs/images/tutorials/transpilation-optimizations-with-sabre/extracted-outputs/eead9bd2-17e0-4f5b-80bc-eb9b30af052e-0.avif" alt="Output of the previous code cell" />

basic: best 2Q depth=524, size=3852 (seed=51)
decay: best 2Q depth=387, size=2786 (seed=45)
lookahead: best 2Q depth=364, size=2485 (seed=51)
StarPreRouting: best 2Q depth=196, size=1151 (seed=42)


In [12]:
# -------------------------Step 3: Execute on hardware-------------------------

best_circuits = {}
for label in ["basic", "decay", "lookahead", "StarPreRouting"]:
    best_circuits[label] = min(
        heuristic_results[label], key=lambda t: t["depth"]
    )
    b = best_circuits[label]
    print(f"Best {label}: 2Q depth={b['depth']}, size={b['size']}")

options = EstimatorOptions()
options.resilience_level = 2
options.dynamical_decoupling.enable = True
options.dynamical_decoupling.sequence_type = "XY4"
estimator = Estimator(backend, options=options)

hw_jobs = {}
hw_ops = {}
for label, best in best_circuits.items():
    hw_ops[label] = [op.apply_layout(best["tqc"].layout) for op in operators]
    hw_jobs[label] = estimator.run([(best["tqc"], hw_ops[label])])
    print(f"{label} job: {hw_jobs[label].job_id()}")
estimator.options.environment.job_tags = ["TUT_TOWS"]

hw_results = {}
for label, job in hw_jobs.items():
    hw_results[label] = job.result()[0]
    print(f"{label} job done")

Best basic: 2Q depth=524, size=3852
Best decay: 2Q depth=387, size=2786
Best lookahead: 2Q depth=364, size=2485
Best StarPreRouting: 2Q depth=196, size=1151
basic job: d81q5tnoha1c73bknprg
decay job: d81q5tugbeec73aktopg
lookahead job: d81q5to0bvlc73d1epe0
StarPreRouting job: d81q5u7tjchs73bn82hg
basic job done
decay job done
lookahead job done
StarPreRouting job done


In [13]:
# -------------------------Step 4: Post-process-------------------------

data = list(range(1, len(operators) + 1))
hw_markers = {
    "basic": "D",
    "decay": "o",
    "lookahead": "s",
    "StarPreRouting": "*",
}
hw_labels = ["basic", "decay", "lookahead", "StarPreRouting"]

fig, (ax1, ax2) = plt.subplots(
    1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [2.5, 1]}
)

# Left: correlations vs distance
for label in hw_labels:
    evs = list(hw_results[label].data.evs)
    b = best_circuits[label]
    ax1.plot(
        data,
        evs,
        marker=hw_markers[label],
        color=hw_colors[label],
        linewidth=2,
        label=f"{label} (2Q depth={b['depth']}, size={b['size']})",
        markersize=5 if label == "StarPreRouting" else 4,
    )

ax1.set_xlabel("Distance between qubits $i$", fontsize=11)
ax1.set_ylabel(r"$\langle Z_0 Z_i \rangle$", fontsize=11)
ax1.set_title(
    "Entanglement correlations vs. qubit distance (hardware)", fontsize=12
)
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

# Right: mean fidelity bar chart
hw_means = [np.mean(list(hw_results[label].data.evs)) for label in hw_labels]
hw_bar_colors = [hw_colors[label] for label in hw_labels]
x_bar = np.arange(len(hw_labels))
bars = ax2.bar(x_bar, hw_means, color=hw_bar_colors)
ax2.set_ylabel(r"Mean $\langle Z_0 Z_i \rangle$", fontsize=11)
ax2.set_title("Average fidelity", fontsize=13)
y_range = (
    max(hw_means) - min(hw_means) if max(hw_means) != min(hw_means) else 0.01
)
ax2.set_ylim(min(hw_means) - y_range * 0.2, max(hw_means) + y_range * 0.15)
for bar, val in zip(bars, hw_means):
    ax2.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + y_range * 0.05,
        f"{val:.4f}",
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold",
    )
ax2.set_xticks(x_bar)
ax2.set_xticklabels(hw_labels, fontsize=9, rotation=15)
ax2.grid(axis="y", linestyle="--", alpha=0.5)

fig.tight_layout()
plt.show()

print("\nMean fidelity:")
for label, m in zip(hw_labels, hw_means):
    print(f"  {label}: {m:.4f}")

<Image src="../docs/images/tutorials/transpilation-optimizations-with-sabre/extracted-outputs/0280b0b9-6320-43e5-8396-f82f9e718319-0.avif" alt="Output of the previous code cell" />


Mean fidelity:
  basic: 0.0344
  decay: 0.1298
  lookahead: 0.1857
  StarPreRouting: 0.3295


### Analysis

The scatter plots show significant variability across seeds for all three SABRE heuristics, which underscores the importance of running multiple layout trials rather than relying on a single transpilation. The `StarPreRouting` line is essentially flat across seeds because the rewrite from a star into a linear chain is deterministic given the structure; the downstream SABRE routing then has very little freedom on a linear chain, so the seed has almost no effect on the final depth or size.

From the transpilation results, both the `decay` and `lookahead` heuristics consistently outperform `basic` by a wide margin. The `basic` heuristic, while fast, uses a simple greedy strategy that often leads to substantially deeper circuits. For this star-topology GHZ circuit, `lookahead` tends to produce the lowest 2Q depth and gate count among the SABRE heuristics, since its forward-looking cost function is well suited to circuits with long-range connectivity patterns. `StarPreRouting`, however, dwarfs all three by a substantial margin: by rewriting the star into a linear chain before routing, it short-circuits the search problem entirely and delivers a circuit that the rest of the transpiler can map onto a linear path with minimal additional SWAPs.

That advantage carries straight over to hardware fidelity. Lower 2Q depth and gate count do not always translate one-for-one to higher fidelity (the specific physical qubits a layout uses and their calibration at run time also matter), but when the depth gap is as large as the one between SABRE and `StarPreRouting` here, the structure-aware approach wins decisively because the circuit accumulates far less decoherence and far fewer two-qubit error events. The fidelity bar chart shows `StarPreRouting` substantially ahead of even the best SABRE heuristic, while `basic` sits well below the rest because its much deeper circuits accumulate the most error.

**Key takeaways:**
- Among SABRE heuristics, `decay` and `lookahead` are substantially better than `basic` for non-trivial circuits. Prefer one of the two for production workloads.
- The best SABRE heuristic depends on your circuit and hardware. Testing multiple heuristics with multiple seeds is the most reliable strategy.
- If you want to explore even more layouts, increase `swap_trials` (and `layout_trials` when you are not pinning a custom routing pass) rather than fanning the work out to remote nodes. The SABRE passes already parallelize trials across local threads, and the per-trial work is small enough that distribution overhead typically dominates any speedup.
- When the circuit has a known special structure, applying a structure-aware pass like `StarPreRouting` before SABRE can deliver an order-of-magnitude improvement that no amount of SABRE tuning will match. This is not a replacement for SABRE: `StarPreRouting` only helps when the circuit actually contains star sub-circuits and the backend has a long enough linear path. It is worth checking the pass library for matches whenever you know your circuit's shape.

## Next steps
If you found this work interesting, you might be interested in the following material:
<Admonition type="tip" title="Recommendations">
- [`SabreLayout` API reference](/docs/api/qiskit/qiskit.transpiler.passes.SabreLayout): full parameter documentation
- [SABRE paper](https://arxiv.org/abs/1809.02573): the original SABRE algorithm for layout and routing
- [LightSABRE paper](https://arxiv.org/abs/2409.08368): the algorithmic improvements that power Qiskit's current SABRE implementation
- [Write a custom transpiler pass](/docs/guides/custom-transpiler-pass): build your own transpilation logic
- [Transpiler plugins](/docs/guides/transpiler-plugins): extend Qiskit's transpilation pipeline with third-party passes
- [DAG representation](/docs/guides/DAG-representation): understand the directed acyclic graph used internally by the transpiler
</Admonition>

## Tutorial survey

Please take this short survey to provide feedback on this tutorial. Your insights will help us improve our content offerings and user experience.

[Link to survey](https://your.feedback.ibm.com/jfe/form/SV_d9YWUSQIAvU9HXE)